In [0]:
# This Cell is used to get Secrets we have in Key Vaultss

client_id = dbutils.secrets.get(scope="kv-scope", key="db-secret-client-id-app-reg")
client_secret = dbutils.secrets.get(scope="kv-scope", key="db-secret-value-appregi")
tenant_id = dbutils.secrets.get(scope="kv-scope", key="db-secret-tenant")

# Storage account name
storage_account = "stdehealthcareanalytics"

# OAuth configs
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
sql_server = "healthcare-project-server-2026.database.windows.net"
sql_user = "username"
sql_pass = dbutils.secrets.get(scope="kv-scope", key="sql-pwd-azureportal")
sql_db = "healthcarebootcamp"

jdbc_url = f"jdbc:sqlserver://{sql_server}:1433;database={sql_db}"

connection_properties = {
    "user": sql_user,
    "password": sql_pass,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
from pyspark.sql.functions import col

# Read FilesTable
files_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.FilesTable",
    properties=connection_properties
)

# Get files ready for Silver
pending_files_df = files_df.filter(
    (col("Status") == "Bronze_Processed") &
    (col("FileName").like("billings%"))
)

# Create list
batch_list = [row.FileName for row in pending_files_df.select("FileName").collect()]

print("Files to process:", batch_list)

# Read files
for file_name in batch_list:
    try:
        print("Reading:", file_name)

        bronze_path = f"abfss://bronze@stdehealthcareanalytics.dfs.core.windows.net/billings/{file_name}"

        df = spark.read.format("parquet").load(bronze_path)

        display(df.limit(10))

    except Exception as e:
        print("Skipped:", file_name, str(e))

Files to process: []


In [0]:
bronze_path = "abfss://bronze@stdehealthcareanalytics.dfs.core.windows.net/billings"
df = spark.read.format("parquet").load(bronze_path)
df.show(100)

+------------------+------------------+------------------+------------+-----------+--------------+---------------+
|        billing_id|        patient_id|          visit_id|billing_date|bill_amount|payment_status| payment_method|
+------------------+------------------+------------------+------------+-----------+--------------+---------------+
|    BIL-15-0000001|    PAT-15-0002885|    VIS-15-0013630|  2015-02-15|     4082.8|        Denied|           null|
|    BIL-15-0000002|    PAT-15-0001362|    VIS-15-0029277|  2016-01-19|     7290.4|       Pending|Insurance Claim|
|    BIL-15-0000003|    PAT-15-0000995|    VIS-15-0028081|  2017-11-25|   11404.48|       Pending|    Credit Card|
|Â BIL-15-0000004\t|    PAT-15-0000709|    VIS-15-0035755|  2016-12-14|      917.2|        Denied|     E-Transfer|
|    BIL-15-0000005|    PAT-15-0000863|    VIS-15-0041548|  2017-04-04|    1177.78|        Unpaid|Insurance Claim|
|              NULL|    PAT-15-0004532|    VIS-15-0021339|  2015-06-01|      607

In [0]:
from pyspark.sql.functions import col, when, trim, regexp_replace, lower
from delta.tables import DeltaTable

silver_path = "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/billings/silver/billings_silver"

print("Starting Silver MERGE")

bronze_path = "abfss://bronze@stdehealthcareanalytics.dfs.core.windows.net/billings"
df = spark.read.format("parquet").load(bronze_path)

# CLEAN IDs (remove Â, quotes, junk)
df = df.select(
    trim(regexp_replace(col("billing_id"), "[^A-Za-z0-9-]", "")).alias("billing_id"),
    trim(regexp_replace(col("patient_id"), "[^A-Za-z0-9-]", "")).alias("patient_id"),
    trim(regexp_replace(col("visit_id"), "[^A-Za-z0-9-]", "")).alias("visit_id"),
    col("billing_date"),
    trim(col("bill_amount")).alias("bill_amount"),
    trim(col("payment_status")).alias("payment_status"),
    trim(col("payment_method")).alias("payment_method")
)

# Better NULL handling
df = df.select([
    when(lower(col(c)).isin("null", ""), None).otherwise(col(c)).alias(c)
    for c in df.columns
])

# Cast amount
df = df.withColumn("bill_amount", col("bill_amount").cast("double"))

# Filter + dedupe
df = df.filter(col("billing_id").isNotNull())
df = df.dropDuplicates(["billing_id"])

# MERGE
from pyspark.sql.utils import AnalysisException

try:
    delta_table = DeltaTable.forPath(spark, silver_path)

    delta_table.alias("target") \
        .merge(df.alias("source"), "target.billing_id = source.billing_id") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

    print("Merge completed")

except:
    # First run → create table
    df.write.format("delta").mode("overwrite").save(silver_path)
    print("Initial load created")

df.display()

Starting Silver MERGE
Initial load created


billing_id,patient_id,visit_id,billing_date,bill_amount,payment_status,payment_method
BIL-15-0000437,PAT-15-0001722,VIS-15-0031862,2017-06-21,4517.76,Denied,Credit Card
BIL-15-0000640,PAT-15-0004845,VIS-15-0094916,2015-01-06,10652.6,null,Credit Card
BIL-15-0000646,PAT-15-0000333,VIS-15-0091435,2016-09-08,1521.13,Pending,Insurance Claim
BIL-15-0000847,PAT-15-0004007,VIS-15-0023866,2016-09-14,10165.12,null,Credit Card
BIL-15-0001360,PAT-15-0000201,VIS-15-0042177,2015-07-02,4503.06,Pending,Credit Card
BIL-15-0001659,PAT-15-0004633,VIS-15-0039393,2017-02-23,3262.39,Unpaid,null
BIL-15-0001843,PAT-15-0000904,VIS-15-0090506,null,5146.93,Denied,Credit Card
BIL-15-0001888,PAT-15-0000886,VIS-15-0074703,2015-09-13,null,Unpaid,Cash
BIL-15-0002051,PAT-15-0004543,VIS-15-0009408,2016-03-06,null,Pending,Credit Card
BIL-15-0002063,null,VIS-15-0006862,2016-12-08,8570.67,null,Insurance Claim


In [0]:
silver_path = "abfss://silver@stdehealthcareanalytics.dfs.core.windows.net/billings/silver/billings_silver"

df = spark.read.format("delta").load(silver_path)

df.display()

billing_id,patient_id,visit_id,billing_date,bill_amount,payment_status,payment_method
BIL-15-0000455,PAT-15-0004864,null,null,null,Denied,null
BIL-15-0000696,PAT-15-0001746,null,2015-08-22,10660.8,Pending,Credit Card
BIL-15-0000967,PAT-15-0001810,null,2015-08-20,null,Unpaid,Cash
BIL-15-0001331,PAT-15-0000211,VIS-15-0037434,2017-05-18,null,Paid,Insurance Claim
BIL-15-0001658,PAT-15-0004143,VIS-15-0057942,2017-05-28,null,Denied,Credit Card
BIL-15-0001661,PAT-15-0002737,null,null,3965.31,Unpaid,null
BIL-15-0001731,PAT-15-0002520,VIS-15-0077212,2015-01-07,8404.8,Paid,Insurance Claim
BIL-15-0001777,PAT-15-0001060,null,2017-09-27,null,Unpaid,Cash
BIL-15-0001945,PAT-15-0003849,VIS-15-0098806,2015-09-03,1041.86,Paid,Cash
BIL-15-0002426,PAT-15-0003729,VIS-15-0002082,2016-06-05,2794.45,Denied,Cash
